# A15 · 多触点归因与随机增量

先修：公用31–34；A14。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

last-click把全部收入记到最后触点；linear按触点均分。它们是记账规则，必须保持总额守恒，却不能回答不投放会发生什么。独立构造随机实验：已知p=.5，潜在结果由同一随机数生成，处理把概率从.10改为.13；展示观测均值差与已知有限总体因果效果。此处无真实propensity、无真实IPS/OPE。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
a=ac.attribution();print(a)
assert np.isclose(sum(a['last_click'].values()),250)
assert np.isclose(sum(a['linear'].values()),250)
assert a['last_click']!=a['linear']
print('随机估计与潜在结果差距',a['randomized_difference']-a['finite_population_effect'])

{'last_click': {'display': 250.0}, 'linear': {'search': 110.0, 'display': 140.0}, 'randomized_difference': 0.03510874558569296, 'finite_population_effect': 0.0315, 'n': 10000, 'assignment_probability': 0.5}
随机估计与潜在结果差距 0.003608745585692963


## 解释与局限

归因分配可以剧烈变化但总收入不变；随机增量是不同问题。随机有限样本估计与已知潜在效果有误差。真实系统需预先定义随机化单位、置信区间、跨组干扰与观察窗口。

## 查看源码，再做练习

改成first-click并保持250总额。构造偏向高意向用户的非随机投放，解释为什么简单均值差不再识别增量。

In [3]:
import inspect
print(inspect.getsource(ac.attribution))

def attribution():
    paths=[(['search','display'],100.),(['display'],60.),(['search','search','display'],90.)]
    last={};linear={}
    for path,revenue in paths:
        last[path[-1]]=last.get(path[-1],0)+revenue
        for channel in path: linear[channel]=linear.get(channel,0)+revenue/len(path)
    assert abs(sum(last.values())-sum(linear.values()))<1e-9
    rng=np.random.default_rng(42); n=10000
    assignment=rng.integers(0,2,n); latent=rng.random(n)
    y0=latent<.10; y1=latent<.13; observed=np.where(assignment,y1,y0)
    return dict(last_click=last,linear=linear,randomized_difference=float(observed[assignment==1].mean()-observed[assignment==0].mean()),
                finite_population_effect=float((y1.astype(int)-y0).mean()),n=n,assignment_probability=.5)



## 来源与下一步

A-RTB 94–116页；已知随机化增量例是Lab自编。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。